# FAISS for TEXT - Quick Start
This notebook is a companion of chapter 2 of the "Domain Specific LLMs in Action" book, author Guglielmo Iozzia, [Manning Publications](https://www.manning.com/), 2024.  
The code in this notebook is to introduce readers to the [FAISS](https://faiss.ai/index.html) library. No hardware acceleration required to execute all the code cells.  

Install the missing required packages in the Colab VM. Only FAISS for CPU, and [SentenceTransformers](https://www.sbert.net/) not available by default.

In [14]:
# !pip install faiss-cpu sentence-transformers

Import the necessary packages/classes.

In [13]:
""" Module to cluster embeddings and create indices. """
import faiss

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

Set the data corpus for this example and put it into a Pandas DataFrame.

In [15]:
data = [["His secret identity is Peter Parker", "spiderman"],
        ["A businessman and engineer who runs the company Start Industries", "ironman"],
        ["Superhuman spider-powers and abilities after being bitten by a radioactive spider", "spiderman"],
        ["A frail man enhanced to the peak of human physical perfection by an experimental super-soldier serum", "captainamerica"]
        ]

df = pd.DataFrame(data, columns=["text", "context"])

In [16]:
df

,text,context
0,His secret identity is Peter Parker,spiderman
1,A businessman and engineer who runs the compan...,ironman
2,Superhuman spider-powers and abilities after b...,spiderman
3,A frail man enhanced to the peak of human phys...,captainamerica


Get embeddings from the data corpus, generate a FAISS index and add the embeddings to it (after normalization).

In [17]:
text = df["text"]
encoder = SentenceTransformer("paraphrase-mpnet-base-v2")
vectors = encoder.encode(text)

In [18]:
vectors

array([[ 0.06908621,  0.40293145, -0.05213062, ...,  0.08801384,
        -0.13445711,  0.06403756],
       [-0.14286482,  0.1006597 , -0.09850159, ..., -0.03079811,
        -0.01277597,  0.01745356],
       [ 0.13080402,  0.07185445,  0.01989626, ...,  0.04210815,
        -0.06912772,  0.05091145],
       [ 0.08117306,  0.14578931,  0.12907588, ..., -0.04734011,
        -0.06134827, -0.04249232]], shape=(4, 768), dtype=float32)

In [19]:
vector_dimension = vectors.shape[1]
print("Vector Dimension:", vector_dimension)
l2_index = faiss.IndexFlatL2(vector_dimension)
print("L2 Index:", l2_index)
faiss.normalize_L2(vectors)
l2_index.add(vectors)

Vector Dimension: 768
L2 Index: <faiss.swigfaiss.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x157fad620> >


In [20]:
vectors[0]

array([ 1.98383462e-02,  1.15703173e-01, -1.49694895e-02,  2.47340444e-02,
        3.79682481e-02, -2.29993910e-02,  6.02637827e-02, -9.17371176e-03,
        1.37601495e-02,  2.50961538e-02, -3.89766172e-02, -2.03392878e-02,
        1.12486612e-02, -3.52051333e-02,  1.80403553e-02, -2.77950726e-02,
        2.56065838e-02, -3.27652283e-02,  3.67175648e-03,  3.50672901e-02,
       -1.94805637e-02,  1.34802051e-02,  7.72050442e-03,  6.37777373e-02,
        4.42951657e-02, -2.71252804e-02,  4.21068594e-02, -4.94873151e-03,
        1.64286382e-02,  2.45893188e-02, -2.36443314e-03, -1.27049424e-02,
       -1.43039636e-02,  3.26322950e-02,  9.29414574e-03,  2.90941782e-02,
        5.52186146e-02,  9.94965341e-03,  4.70733084e-03, -1.79625656e-02,
       -5.42631000e-02,  2.99972612e-02,  2.70248782e-02, -1.41599146e-03,
       -1.08192768e-02, -3.85977030e-02,  4.21064571e-02,  5.43395244e-02,
       -5.73823377e-02,  2.10921522e-02,  2.65319627e-02,  1.32841067e-02,
        3.84260677e-02, -

Prepare a search text to be used for similarity search with FAISS on the generated index.

In [21]:
search_text = "He throws webs"
search_vector = encoder.encode(search_text)
search_vector_as_array = np.array([search_vector])
faiss.normalize_L2(search_vector_as_array)

Perform a search within the created index (calculation of the distances between the search text and the strings within the index).

In [22]:
k = l2_index.ntotal
distances, ann = l2_index.search(search_vector_as_array, k=k)

In [23]:
distances

array([[1.5010695, 1.552392 , 1.642769 , 1.731641 ]], dtype=float32)

In [24]:
ann

array([[2, 0, 1, 3]])

Prepare the results to be displayed in a user-friendly format.

In [25]:
search_results = pd.DataFrame({"distances": distances[0], "ann": ann[0]})
merged_df = pd.merge(search_results, df, left_on="ann", right_index=True)
merged_df.head()

,distances,ann,text,context
0,1.501070,2,Superhuman spider-powers and abilities after b...,spiderman
1,1.552392,0,His secret identity is Peter Parker,spiderman
2,1.642769,1,A businessman and engineer who runs the compan...,ironman
3,1.731641,3,A frail man enhanced to the peak of human phys...,captainamerica
